<a href="https://colab.research.google.com/github/msoley/CSCI-535/blob/main/multimodal_fusion_practicum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Fusion Techniques for Emotion Recognition
### CSCI-535 — Multimodal Probabilistic Learning of Human Communication

In this practicum, we implement and compare **four fusion strategies** for audiovisual emotion recognition using (a subset of) [CREMA-D](https://github.com/CheyneyComputerScience/CREMA-D) dataset. Actors portray six emotions (happiness, sadness, anger, fear, disgust, neutral) in both voice and facial expression.

We use two pre-extracted feature sets:
- **OpenFace** (visual, 20-dim): facial action unit intensities and head pose angles
- **HuBERT** (audio, 768-dim): penultimate-layer representations from HuBERT-base, temporally pooled ×5

All models are implemented in **PyTorch**.

---

| # | Strategy | Where | Key Idea |
|---|----------|--------|----------|
| 1 | **Early Fusion** | Input | Concatenate features before the model |
| 2 | **Late Fusion** | Output | Combine predictions from separate models |
| 3 | **Neural Concatenation** | Representation | Join learned hidden states mid-network |



## Learning Objectives

By the end of this practicum you will be able to:
1. Build a `torch.utils.data.Dataset` for sequential feature files and load it with `DataLoader`
2. Implement a GRU classifier with `nn.Module` and write a manual training loop in PyTorch
3. Apply **early fusion** by concatenating features at the input
4. Apply **late fusion** — both uniform averaging and a learned blend weight
5. Build a **neural concatenation fusion** model with two parallel GRU branches
6. Compare all strategies and articulate their trade-offs

### PyTorch core concepts you will see
| Concept | Where used |
|---------|------------|
| `Dataset` / `DataLoader` | All sections |
| `nn.Module` subclassing | Sections 3–7 |
| `model.train()` / `model.eval()` | All training loops |
| `optimizer.zero_grad()` / `loss.backward()` / `optimizer.step()` | All training loops |
| `torch.no_grad()` | Evaluation |
| `.to(device)` for GPU | Throughout |
| Manual early stopping | Training helper |


---
## Section 0: Setup


In [ ]:
!pip install -q tqdm seaborn scikit-learn

In [ ]:
# ── Download and extract pre-extracted features ──────────────────────────────
import os, zipfile

if not os.path.isdir('subsampled'):
    !wget -q --show-progress https://people.ict.usc.edu/~soleymani/subsampled.zip
    with zipfile.ZipFile('subsampled.zip', 'r') as z:
        z.extractall('.')
    os.remove('subsampled.zip')
    print('Done.')
else:
    print('Features already extracted.')

# ── Feature paths and split file lists ───────────────────────────────────────
FEAT_PATH_V = 'subsampled/openface'   # OpenFace AUs  (train/val/test subdirs)
FEAT_PATH_A = 'subsampled/hubert'     # HuBERT pooled (train/val/test subdirs)

splits = {}
for split in ['train', 'val', 'test']:
    splits[split] = sorted([f for f in os.listdir(os.path.join(FEAT_PATH_V, split))
                            if f.endswith('.npy')])
    print(f'{split:6s}: {len(splits[split])} utterances')


In [ ]:
import numpy as np
import os, random, copy
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ──────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

# ── Constants ───────────────────────────────────────────────────────────────
FEAT_V        = 20    # OpenFace: 17 AU intensities + 3 head pose angles
FEAT_A        = 768   # HuBERT-base penultimate layer (5× temporal pool)
MAX_LEN_V     = 16    # visual sequence length  (OpenFace, median ~16 frames)
MAX_LEN_A     = 40    # audio sequence length   (HuBERT pooled, median ~32)
MAX_LEN_EARLY = 40    # common length for early fusion
NUM_CLASSES   = 6
GRU_UNITS     = 64
EPOCHS        = 15
BATCH_SIZE    = 32
LR            = 1e-3

labels_dict = {'ANG': 0, 'DIS': 1, 'FEA': 2, 'HAP': 3, 'NEU': 4, 'SAD': 5}
label_names = ['Anger', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness']

results_table = {}   # collect test accuracy for every method


---
## Section 1: Feature Exploration

**OpenFace features (20 dimensions):**
```
AU01_r  AU02_r  AU04_r  AU05_r  AU06_r  AU07_r  AU09_r  AU10_r
AU12_r  AU14_r  AU15_r  AU17_r  AU20_r  AU23_r  AU25_r  AU26_r
AU45_r  pose_Rx  pose_Ry  pose_Rz
```
See the [FACS reference](https://www.cs.cmu.edu/~face/facs.htm) for action unit definitions.  
**AU04** = Brow Lowerer — furrowed brows, common in anger/fear  
**AU06** = Cheek Raiser — Duchenne smile marker, genuine happiness

**HuBERT features (768-dim):**  
Penultimate transformer layer (layer 11/12) of HuBERT-base trained on LibriSpeech 960h.  
Originally at 50 Hz (20 ms hop); temporally pooled ×5 → ~10 Hz effective rate.


In [ ]:
def extract_label(fname):
    return fname.split('_')[2]

train_files = splits['train']
pooled, labels_viz = [], []
for fname in tqdm(train_files):
    data  = np.load(os.path.join(FEAT_PATH_V, 'train', fname))
    label = extract_label(fname)
    if label in ['ANG', 'SAD', 'HAP']:
        labels_viz.append(label)
        pooled.append(data[:, [2, 4]].mean(axis=0))   # AU04, AU06 averaged over time

pooled = np.clip(np.array(pooled), 0, None)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(ax=axes[0], x=pooled[:, 0], y=pooled[:, 1], hue=labels_viz, alpha=0.6)
axes[0].set_xlabel('AU04 (Brow Lowerer)')
axes[0].set_ylabel('AU06 (Cheek Raiser)')
axes[0].set_title('OpenFace: AU04 vs AU06')

for emo, col in zip(['ANG', 'HAP', 'SAD'], ['tomato', 'seagreen', 'steelblue']):
    idx = [i for i, l in enumerate(labels_viz) if l == emo]
    axes[1].hist(pooled[idx, 0], bins=20, alpha=0.5, label=emo, color=col)
axes[1].set_xlabel('AU04 Intensity'); axes[1].set_title('AU04 distribution per class')
axes[1].legend()
plt.tight_layout(); plt.show()


> **Q1.1** Which emotion classes are separable in AU04 vs AU06 space? Which are not?  Is there any quantitiave measure we could use for feature utility?


---
## Section 2: PyTorch Data Loading

In PyTorch, data loading is split across two classes:

| Class | Responsibility |
|-------|----------------|
| `torch.utils.data.Dataset` | Knows how to load and transform **one** sample |
| `torch.utils.data.DataLoader` | Wraps a Dataset; handles batching, shuffling, and parallel workers |

Your `Dataset` must implement:
- `__len__(self)` → total number of samples
- `__getitem__(self, idx)` → the sample at index `idx` as tensors

### Padding strategy
Each `.npy` file contains a `(T, feature_dim)` array where `T` varies per clip.
We **zero-pad** shorter sequences to `MAX_LEN` and clip longer ones.  
The GRU sees zeros at the padded positions (a minor approximation; see the extension note at the end).

In [ ]:
class SequenceDataset(Dataset):
    """Single-modality sequential feature dataset.

    Args:
        data_dir:    directory containing .npy feature files
        file_list:   list of filenames to include (speaker split)
        labels_dict: mapping from emotion code string to integer label
        n_channels:  feature dimension of each timestep
        target_size: fixed sequence length (pad / clip to this)
    """
    def __init__(self, data_dir, file_list, labels_dict, n_channels, target_size):
        self.data_dir    = data_dir
        self.n_channels  = n_channels
        self.target_size = target_size
        self.files  = sorted(file_list)
        self.labels = [labels_dict[f.split('_')[2]] for f in self.files]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        x = np.load(os.path.join(self.data_dir, self.files[idx])).astype(np.float32)
        T = x.shape[0]
        if T > self.target_size:
            x = x[:self.target_size]
        elif T < self.target_size:
            x = np.vstack([x, np.zeros((self.target_size - T, self.n_channels), np.float32)])
        return torch.tensor(x), self.labels[idx]  # (target_size, features), int


class BimodalDataset(Dataset):
    """Two-modality dataset; yields (x_visual, x_audio, label).

    Both modality directories must contain identically named .npy files.
    Each modality is padded/clipped to its own target size.

    Args:
        path_v, path_a:           directories for visual and audio features
        file_list:                list of filenames (shared across modalities)
        feat_v, feat_a:           feature dimensions
        target_size_v/a:          sequence lengths for each modality
    """
    def __init__(self, path_v, path_a, file_list, labels_dict,
                 feat_v, feat_a, target_size_v, target_size_a):
        self.path_v         = path_v
        self.path_a         = path_a
        self.feat_v         = feat_v
        self.feat_a         = feat_a
        self.target_size_v  = target_size_v
        self.target_size_a  = target_size_a
        self.files  = sorted(file_list)
        self.labels = [labels_dict[f.split('_')[2]] for f in self.files]

    def _load(self, path, fname, feat_dim, target_size):
        x = np.load(os.path.join(path, fname)).astype(np.float32)
        T = x.shape[0]
        if T > target_size:
            x = x[:target_size]
        elif T < target_size:
            x = np.vstack([x, np.zeros((target_size - T, feat_dim), np.float32)])
        return torch.tensor(x)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        x_v = self._load(self.path_v, fname, self.feat_v, self.target_size_v)
        x_a = self._load(self.path_a, fname, self.feat_a, self.target_size_a)
        return x_v, x_a, self.labels[idx]


class MergedDataset(Dataset):
    """Early-fusion dataset: pads both modalities to MAX_LEN_EARLY and
    concatenates along the feature axis → (MAX_LEN_EARLY, FEAT_V + FEAT_A).
    """
    def __init__(self, path_v, path_a, file_list, labels_dict,
                 feat_v, feat_a, target_size):
        self.bm = BimodalDataset(path_v, path_a, file_list, labels_dict,
                                 feat_v, feat_a, target_size, target_size)

    def __len__(self):
        return len(self.bm)

    def __getitem__(self, idx):
        x_v, x_a, label = self.bm[idx]
        return torch.cat([x_v, x_a], dim=-1), label  # (T, FEAT_V+FEAT_A)


**Q2.1** Is this the best approach? What heppens to the padded sequences?

In [ ]:
def make_loaders(feat_base, feat_dim, max_len):
    """Create train/val/test DataLoaders for a single modality.
    feat_base must contain train/val/test subdirs.
    """
    loaders = {}
    for split, shuffle in [('train', True), ('val', False), ('test', False)]:
        split_dir = os.path.join(feat_base, split)
        file_list = sorted([f for f in os.listdir(split_dir) if f.endswith('.npy')])
        ds = SequenceDataset(split_dir, file_list, labels_dict, feat_dim, max_len)
        loaders[split] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                                    num_workers=0, pin_memory=(device.type == 'cuda'))
    return loaders['train'], loaders['val'], loaders['test']


def make_bimodal_loaders():
    """Create train/val/test DataLoaders for both modalities simultaneously."""
    loaders = {}
    for split, shuffle in [('train', True), ('val', False), ('test', False)]:
        ds = BimodalDataset(
            path_v=os.path.join(FEAT_PATH_V, split),
            path_a=os.path.join(FEAT_PATH_A, split),
            file_list=splits[split], labels_dict=labels_dict,
            feat_v=FEAT_V, feat_a=FEAT_A,
            target_size_v=MAX_LEN_V, target_size_a=MAX_LEN_A)
        loaders[split] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                                    num_workers=0, pin_memory=(device.type == 'cuda'))
    return loaders['train'], loaders['val'], loaders['test']


def make_merged_loaders():
    """Create DataLoaders for early fusion (concatenated features, MAX_LEN_EARLY)."""
    loaders = {}
    for split, shuffle in [('train', True), ('val', False), ('test', False)]:
        ds = MergedDataset(
            path_v=os.path.join(FEAT_PATH_V, split),
            path_a=os.path.join(FEAT_PATH_A, split),
            file_list=splits[split], labels_dict=labels_dict,
            feat_v=FEAT_V, feat_a=FEAT_A, target_size=MAX_LEN_EARLY)
        loaders[split] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                                    num_workers=0, pin_memory=(device.type == 'cuda'))
    return loaders['train'], loaders['val'], loaders['test']


# Quick sanity check
train_of, val_of, test_of = make_loaders(FEAT_PATH_V, FEAT_V, MAX_LEN_V)
X_batch, y_batch = next(iter(train_of))
print(f'Unimodal OpenFace batch — X: {X_batch.shape}, y: {y_batch.shape}')

train_bm, val_bm, test_bm = make_bimodal_loaders()
xv, xa, yb = next(iter(train_bm))
print(f'Bimodal batch           — visual: {xv.shape}, audio: {xa.shape}, y: {yb.shape}')

train_ef, val_ef, test_ef = make_merged_loaders()
xm, ym = next(iter(train_ef))
print(f'Early fusion batch      — merged: {xm.shape}, y: {ym.shape}')


---
## Section 3: Unimodal Baselines

### 3.1  Training and evaluation helpers

In PyTorch there is no `.fit()`. We write the training loop ourselves.  
This gives you full control but requires a few boilerplate steps every epoch:

```
for each batch:
    optimizer.zero_grad()   # clear gradients from previous step
    logits = model(X)       # forward pass
    loss = criterion(logits, y)
    loss.backward()         # compute gradients
    optimizer.step()        # update weights
```

Key rules:
- Call `model.train()` before the training loop and `model.eval()` before validation
- Wrap validation/inference in `torch.no_grad()` to skip gradient bookkeeping
- `nn.CrossEntropyLoss` expects **raw logits** (no softmax), not probabilities

In [ ]:
criterion = nn.CrossEntropyLoss()

def run_epoch(model, loader, optimizer=None, bimodal=False):
    """One training or evaluation epoch.
    
    If optimizer is None → evaluation mode (no gradient updates).
    bimodal=True → loader yields (x_v, x_a, y) instead of (x, y).
    Returns (mean_loss, accuracy).
    """
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, n = 0.0, 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            if bimodal:
                x_v, x_a, y = batch
                x_v, x_a, y = x_v.to(device), x_a.to(device), y.to(device)
                logits = model(x_v, x_a)
            else:
                x, y = batch
                x, y = x.to(device), y.to(device)
                logits = model(x)

            loss = criterion(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(y)
            correct    += (logits.argmax(1) == y).sum().item()
            n          += len(y)

    return total_loss / n, correct / n


def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR,
                patience=4, bimodal=False):
    """Full training loop with early stopping (monitors val accuracy)."""
    optimizer   = torch.optim.Adam(model.parameters(), lr=lr)
    best_acc    = 0.0
    best_state  = None
    no_improve  = 0
    history     = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, bimodal)
        vl_loss, vl_acc = run_epoch(model, val_loader,   None,      bimodal)

        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss);   history['val_acc'].append(vl_acc)

        print(f'  Epoch {epoch+1:02d}/{epochs}  '
              f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  '
              f'val_loss={vl_loss:.4f}  val_acc={vl_acc:.3f}')

        if vl_acc > best_acc:
            best_acc   = vl_acc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)   # restore best checkpoint
    return history


@torch.no_grad()
def evaluate(model, loader, name, bimodal=False):
    """Collect predictions, compute accuracy, update results_table."""
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        if bimodal:
            x_v, x_a, y = batch
            logits = model(x_v.to(device), x_a.to(device))
        else:
            x, y = batch
            logits = model(x.to(device))
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_labels.append(y.numpy())
    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    acc    = accuracy_score(labels, preds) * 100
    results_table[name] = acc
    print(f'[{name}] Test accuracy: {acc:.2f}%')
    return preds, labels

### 3.2  Exercise — Build the GRU Classifier

Complete `GRUClassifier` below. The target architecture:

```
Input  (batch, MAX_LEN, feature_dim)
  → nn.GRU(feature_dim, hidden_size, batch_first=True)
  → take last hidden state  h_n[-1]  → shape (batch, hidden_size)
  → nn.Linear(hidden_size, hidden_size) + ReLU
  → nn.Dropout(0.3)
  → nn.Linear(hidden_size, num_classes)   ← raw logits, NO softmax
```

> **Why no softmax here?**  `nn.CrossEntropyLoss` already applies `log_softmax` internally.  
> At inference, use `logits.argmax(dim=1)` for the predicted class.

In [ ]:
class GRUClassifier(nn.Module):
    """
    Unimodal GRU sequence classifier.

    Args:
        input_size:  feature dimension per timestep
        hidden_size: number of GRU hidden units
        num_classes: number of output classes
    """
    def __init__(self, input_size, hidden_size=GRU_UNITS, num_classes=NUM_CLASSES):
        super().__init__()
        self.gru  = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc1  = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.3)
        self.fc2  = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, h_n = self.gru(x)     # h_n: (1, batch, hidden)
        h = h_n[-1]              # (batch, hidden)
        return self.fc2(self.drop(self.relu(self.fc1(h))))


# Smoke test
model_test = GRUClassifier(FEAT_V).to(device)
dummy = torch.zeros(4, MAX_LEN_V, FEAT_V).to(device)
out   = model_test(dummy)
assert out.shape == (4, NUM_CLASSES), f'Unexpected output shape: {out.shape}'
print('Shape check passed:', out.shape)


In [ ]:
# ── Train: Unimodal OpenFace (visual) ───────────────────────────────────────
train_of, val_of, test_of = make_loaders(FEAT_PATH_V, FEAT_V, MAX_LEN_V)

print('=== Unimodal: OpenFace (visual) ===')
model_AU = GRUClassifier(FEAT_V).to(device)
history_AU = train_model(model_AU, train_of, val_of)


In [ ]:
# ── Train: Unimodal HuBERT (audio) ─────────────────────────────────────────
train_hub, val_hub, test_hub = make_loaders(FEAT_PATH_A, FEAT_A, MAX_LEN_A)

print('=== Unimodal: HuBERT (audio) ===')
model_hub = GRUClassifier(FEAT_A).to(device)
history_hub = train_model(model_hub, train_hub, val_hub)


In [ ]:
preds_AU, labels_AU = evaluate(model_AU, test_of,  'Unimodal OpenFace')
preds_hub, labels_hub = evaluate(model_hub, test_hub, 'Unimodal HuBERT')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for hist, name, ax in zip([history_AU, history_hub], ['OpenFace', 'HuBERT'], axes):
    ax.plot(hist['train_acc'], label='train')
    ax.plot(hist['val_acc'],   label='val')
    ax.set_title(f'{name} accuracy'); ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


**Q3.1** What do you see in the learning curves? 

**Q3.2** Can you turn the GRU to a Bi-GRU? What difference does it make?

---
## Section 4: Early Fusion

**Idea:** Concatenate all modality features at the **input** before any learned layer.  
A single model then learns from the full `(T, 20+768)` sequence.

```
OpenFace (T,  20) ──┐
                     concatenate → (T, 788) → GRU → Dense → logits
HuBERT   (T, 768) ──┘
```

Both sequences are padded to `MAX_LEN_EARLY=40` before concatenation.

### Exercise 4.1 — Build the early fusion model

The input is now 788-dimensional. Consider using `hidden_size=128` and a
second stacked GRU layer to handle the richer input.


In [ ]:
class EarlyFusionGRU(nn.Module):
    """
    Early fusion GRU: input is the concatenation of visual + audio features.
    input_size = FEAT_V + FEAT_A = 788
    """
    def __init__(self, input_size=FEAT_V + FEAT_A,
                 hidden_size=128, num_classes=NUM_CLASSES):
        super().__init__()
        self.gru  = nn.GRU(input_size, hidden_size, num_layers=2,
                           batch_first=True, dropout=0.3)
        self.fc1  = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.3)
        self.fc2  = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, h_n = self.gru(x)
        return self.fc2(self.drop(self.relu(self.fc1(h_n[-1]))))


train_ef, val_ef, test_ef = make_merged_loaders()

print('=== Early Fusion ===')
model_ef = EarlyFusionGRU().to(device)
history_ef = train_model(model_ef, train_ef, val_ef)
preds_ef, labels_ef = evaluate(model_ef, test_ef, 'Early Fusion')


> **Q4.1** How does the early fusion model compare to the two unimodal baselines? Did you expect this?  
> **Q4.2** The audio features (512-dim) dominate the concatenated vector. How might this bias the model?  
> **Q4.3** What pre-processing step could reduce the dimensionality imbalance before concatenation?

---
## Section 5: Late Fusion

**Idea:** Train separate unimodal models (already done in Section 3), then combine their **softmax probability vectors** at inference time. No new model is trained for the fusion itself.

```
OpenFace → GRU → softmax ──┐
                             weighted sum → argmax → prediction
TRILL    → GRU → softmax ──┘
```

### Exercise 5.1 — Uniform average

Implement `late_fusion_predict` using equal weights (0.5 / 0.5).

In [ ]:
@torch.no_grad()
def get_probs(model, loader, bimodal=False):
    """Return softmax probabilities and labels for an entire DataLoader."""
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        if bimodal:
            x_v, x_a, y = batch
            logits = model(x_v.to(device), x_a.to(device))
        else:
            x, y = batch
            logits = model(x.to(device))
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


def late_fusion_predict(probs_v, probs_a, w=0.5):
    """Combine predictions by weighted average of softmax probabilities.
    w = weight for visual, (1-w) = weight for audio.
    """
    combined = w * probs_v + (1 - w) * probs_a
    return combined.argmax(axis=1)


# Pre-compute probabilities once (reused in grid search)
val_probs_AU, val_labels   = get_probs(model_AU, val_of)
val_probs_hub, _            = get_probs(model_hub, val_hub)
test_probs_AU, test_labels = get_probs(model_AU, test_of)
test_probs_hub, _           = get_probs(model_hub, test_hub)

# Uniform average
lf_preds_unif = late_fusion_predict(test_probs_AU, test_probs_hub, w=0.5)
acc_lf_unif   = accuracy_score(test_labels, lf_preds_unif) * 100
results_table['Late Fusion (avg)'] = acc_lf_unif
print(f'[Late Fusion — uniform avg] Test accuracy: {acc_lf_unif:.2f}%')


### Exercise 5.2 — Learned blend weight

Sweep over possible values of `w ∈ [0, 1]` on the **validation** set and pick the weight that maximises validation accuracy. Apply that weight on the **test** set.

In [ ]:
# ── Grid search over w on validation set ────────────────────────────────────
best_w, best_val_acc = 0.5, 0.0
alphas, val_accs = [], []

for w in np.linspace(0, 1, 21):
    preds = late_fusion_predict(val_probs_AU, val_probs_hub, w=w)
    acc   = accuracy_score(val_labels, preds)
    alphas.append(w); val_accs.append(acc * 100)
    if acc > best_val_acc:
        best_val_acc, best_w = acc, w

print(f'Best w (visual weight): {best_w:.2f}  val_acc={best_val_acc*100:.2f}%')

# Apply best w to test set
lf_preds_w = late_fusion_predict(test_probs_AU, test_probs_hub, w=best_w)
acc_lf_w   = accuracy_score(test_labels, lf_preds_w) * 100
results_table['Late Fusion (learned w)'] = acc_lf_w
print(f'[Late Fusion — w={best_w:.2f}] Test accuracy: {acc_lf_w:.2f}%')

plt.figure(figsize=(7, 4))
plt.plot(alphas, val_accs, marker='o')
plt.axvline(best_w, color='red', ls='--', label=f'best w={best_w:.2f}')
plt.xlabel('w  (weight for AU / visual)')
plt.ylabel('Val Accuracy (%)')
plt.title('Late Fusion: weight sweep on validation set')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


---
## Section 6: Neural Concatenation Fusion (Intermediate Fusion)

**Idea:** Give each modality its own GRU encoder. After the GRU extracts a compact representation from each, **concatenate the hidden states** and pass them through shared FC layers. The whole model is trained jointly end-to-end.

```
OpenFace (B, T, 20)  → GRU → h_v  (B, H) ──┐
                                              Cat(B, 2H) → FC → logits
TRILL    (B, T,512)  → GRU → h_a  (B, H) ──┘
```

This model takes **two tensors** as input, so the `forward` method signature is `forward(self, x_v, x_a)`.

### Exercise 6.1 — Implement NeuralConcatFusion

In [ ]:
class NeuralConcatFusion(nn.Module):
    """
    Dual-branch GRU with intermediate (hidden-state) concatenation.
    """
    def __init__(self, feat_v=FEAT_V, feat_a=FEAT_A,
                 hidden_size=GRU_UNITS, num_classes=NUM_CLASSES):
        super().__init__()
        self.gru_v = nn.GRU(feat_v, hidden_size, batch_first=True)
        self.gru_a = nn.GRU(feat_a, hidden_size, batch_first=True)
        self.fc1   = nn.Linear(hidden_size * 2, hidden_size)
        self.relu  = nn.ReLU()
        self.drop  = nn.Dropout(0.3)
        self.fc2   = nn.Linear(hidden_size, num_classes)

    def forward(self, x_v, x_a):
        _, h_v = self.gru_v(x_v); h_v = h_v[-1]
        _, h_a = self.gru_a(x_a); h_a = h_a[-1]
        h = torch.cat([h_v, h_a], dim=-1)
        return self.fc2(self.drop(self.relu(self.fc1(h))))


# Shape check
model_nc_test = NeuralConcatFusion().to(device)
dv = torch.zeros(4, MAX_LEN_V, FEAT_V).to(device)
da = torch.zeros(4, MAX_LEN_A, FEAT_A).to(device)
assert model_nc_test(dv, da).shape == (4, NUM_CLASSES)
print('Shape check passed')

print('\n=== Neural Concatenation Fusion ===')
model_nc = NeuralConcatFusion().to(device)
history_nc = train_model(model_nc, train_bm, val_bm, bimodal=True)
preds_nc, labels_nc = evaluate(model_nc, test_bm, 'Neural Concat Fusion', bimodal=True)


---
## Section 7: Results Comparison


In [ ]:
print('\n' + '='*52)
print(f'{"Method":<32} {"Test Accuracy":>12}')
print('='*52)
for method, acc in sorted(results_table.items(), key=lambda x: x[1]):
    print(f'{method:<32} {acc:>10.2f}%')
print('='*52)

methods = list(results_table.keys())
accs    = [results_table[m] for m in methods]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(methods, accs, edgecolor='black', linewidth=0.5)
axes[0].set_ylim(0, 100)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{acc:.1f}%', ha='center', va='bottom', fontsize=9)
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Fusion Strategy Comparison')
axes[0].tick_params(axis='x', rotation=30)

# Confusion matrix for best model
best_method = max(results_table, key=results_table.get)
preds_map = {
    'Unimodal AU':      (preds_AU,  labels_AU),
    'Unimodal HuBERT':         (preds_hub,  labels_hub),
    'Early Fusion':           (preds_ef,  labels_ef),
    'Neural Concat Fusion':   (preds_nc,  labels_nc),
    'Gradient Blending':      (preds_gb,  labels_gb),
}
if best_method in preds_map:
    p, l = preds_map[best_method]
    cm = confusion_matrix(l, p)
    ConfusionMatrixDisplay(cm, display_labels=label_names).plot(
        ax=axes[1], cmap='Blues', colorbar=False)
    axes[1].set_title(f'Confusion Matrix — {best_method}')
    axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout(); plt.show()

---
## Section 7: Discussion Questions

**7.1 — Fusion trade-offs**
- Which method performed best? Did the result surprise you?
- What are the computational trade-offs (# parameters, training time) between methods?
- In what real-world scenario would you prefer **late fusion** even if another method is more accurate?


**7.4 — Extensions (bonus)**
- **Attention fusion:** Replace the hidden-state concatenation in `NeuralConcatFusion` with a cross-modal attention mechanism. Sketch the new `forward()` method.
- **Missing modality robustness:** How would each of the four methods handle a test sample where audio is completely missing (e.g., a silent video clip)?
- **Develop modality dropout for robut fusion. When is it helpful?
